# PVGIS-only ST-GNN — Huber + MC-Dropout + SDE-proxy pipeline

Reproducible orchestrator for the final run on server **newzealand**.

Does **not** duplicate runner/analysis logic — it builds commands and reads the
CSVs they write, via `physiq_pv.experiments.sde_proxy_pipeline`.

Safety switches: `RUN_TRAINING`, `RUN_ANALYSIS`, `RUN_SWEEP`, `CREATE_SWEEP`, `LOG_TO_WANDB` — all default off.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)

from physiq_pv.experiments import sde_proxy_pipeline as pipe
print('repo root:', REPO_ROOT)

In [ ]:
PVGIS_DIR = pipe.PVGIS_DIR
TEST_ANOMALY_SCORES = pipe.TEST_ANOMALY_SCORES
TRAIN_ANOMALY_SCORES = pipe.TRAIN_ANOMALY_SCORES
ANALYSIS_SCRIPT = pipe.ANALYSIS_SCRIPT

checks = {
    'PVGIS dir': Path(PVGIS_DIR).is_dir(),
    'test anomaly scores': Path(TEST_ANOMALY_SCORES).exists(),
    'train anomaly scores': Path(TRAIN_ANOMALY_SCORES).exists(),
    'analysis script': Path(ANALYSIS_SCRIPT).exists(),
}
try:
    import physiq_pv.experiments.pvgis_stgnn_runner  # noqa: F401
    checks['runner importable'] = True
except Exception as e:
    checks['runner importable'] = False
    print('runner import error:', e)
for k, v in checks.items():
    print(('OK     ' if v else 'MISSING') + '  ' + k)

If the anomaly scores are **missing**, regenerate them (run in a terminal — not launched automatically):

In [ ]:
if not (Path(TEST_ANOMALY_SCORES).exists() and Path(TRAIN_ANOMALY_SCORES).exists()):
    base = ('PYTHONPATH=$PWD python scripts/run_pvgis_climatology_anomaly_years.py'
            ' --pvgis-dir ' + PVGIS_DIR +
            ' --climatology-start-year 2005 --climatology-end-year 2023'
            ' --quantile 0.975 --climatology-window-days 15 --min-climatology-years 3'
            ' --variables solar_irradiance_poa pv_power_output temperature_2m wind_speed_10m'
            ' --out-root outputs')
    print('# Test year 2019:')
    print(base + ' --years 2019')
    print()
    print('# Train years 2016-2018 (aggregated):')
    print(base + ' --years 2016,2017,2018 --aggregate-out-dir ' + str(Path(TRAIN_ANOMALY_SCORES).parent))
else:
    print('anomaly scores present.')

## 2. Single-run config

In [ ]:
BASE_CONFIG = dict(pipe.DEFAULT_CONFIG)  # edit to override

out_dir = pipe.make_out_dir(BASE_CONFIG)
run_name = pipe.make_run_name(BASE_CONFIG)
print('out_dir  :', out_dir)
print('run_name :', run_name)
if Path(out_dir).exists():
    print('WARNING: out_dir already exists — a run would overwrite its files.')

## 3. Training command

In [ ]:
train_cmd = pipe.build_train_command(
    BASE_CONFIG, out_dir=out_dir, run_name=run_name,
    pvgis_dir=PVGIS_DIR, test_anomaly_scores=TEST_ANOMALY_SCORES,
    train_anomaly_scores=TRAIN_ANOMALY_SCORES, device='cuda', use_wandb=True)
print(' \\\n  '.join(train_cmd))

## 4. Run training

Set `RUN_TRAINING = True` to actually launch.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(train_cmd, check=True)
else:
    print('RUN_TRAINING is False — not launching. Command above is what would run.')

## 5. Post-hoc analysis

In [ ]:
RUN_ANALYSIS = False

analysis_cmd = pipe.build_analysis_command(out_dir, BASE_CONFIG)
print(' \\\n  '.join(analysis_cmd))
if RUN_ANALYSIS:
    subprocess.run(analysis_cmd, check=True)
else:
    print('RUN_ANALYSIS is False — not launching.')

## 6. Read results

In [ ]:
def _read(name):
    p = Path(out_dir) / name
    return pd.read_csv(p) if p.exists() else None

RESULT_FILES = ['metrics_global.csv','metrics_daytime.csv','metrics_by_anomaly_label.csv',
                'residual_bias_and_bin_metrics.csv','daytime_bin_summary.csv',
                'daytime_bin_anomaly_metrics.csv','uncertainty_response.csv','sharpness_overview.csv']
results = {n: _read(n) for n in RESULT_FILES}
for n, df in results.items():
    print((('OK  ' if df is not None else '--  ') + n) + (('  ' + str(df.shape)) if df is not None else ''))

In [ ]:
sharp = results['sharpness_overview.csv']
if sharp is not None:
    cols = [c for c in ['scope','count','picp','mae','rmse','mean_std','mpiw','nmpil'] if c in sharp.columns]
    display(sharp[cols])
bins = results['daytime_bin_summary.csv']
if bins is not None:
    display(bins)
unc = results['uncertainty_response.csv']
if unc is not None:
    display(unc)

## 7. Plots

`matplotlib` + `pandas` only. Figures saved under `<out_dir>/figures/`.
`predictions.csv` is large — sampled via `pipe.load_prediction_sample`.

In [ ]:
MAX_PLOT_ROWS = 500_000
FIG_DIR = Path(out_dir) / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = Path(out_dir) / 'predictions.csv'

pred = None
if PRED_PATH.exists():
    pred = pipe.load_prediction_sample(PRED_PATH, max_rows=MAX_PLOT_ROWS, random_state=1)
    ycol = 'y_pred_mean' if 'y_pred_mean' in pred.columns else 'y_pred'
    pred['residual'] = pred['y_true'] - pred[ycol]
    pred['abs_error'] = pred['residual'].abs()
    pred['interval_width'] = pred['upper_pi'] - pred['lower_pi']
    def _bin(y):
        for name, lo, hi in pipe.PRODUCTION_BINS:
            if y >= lo and (hi is None or y < hi):
                return name
        return 'other'
    pred['prod_bin'] = pred['y_true'].apply(_bin)
    print('plot sample rows:', len(pred))
else:
    print('predictions.csv not found — run training first.')

In [ ]:
def _save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=120, bbox_inches='tight'); plt.close(fig)
    print('saved', path); return path

BIN_ORDER = [b[0] for b in pipe.PRODUCTION_BINS]
figure_paths = {}
if pred is not None:
    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(pred['residual'].dropna(), bins=100); ax.set_title('Residual (y_true - y_pred_mean)')
    ax.set_xlabel('residual [W]'); ax.set_ylabel('count')
    figure_paths['residual_histogram'] = _save(fig, 'residual_histogram.png')

    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(pred['interval_width'].dropna(), bins=100); ax.set_title('Interval width (upper_pi - lower_pi)')
    ax.set_xlabel('interval width [W]'); ax.set_ylabel('count')
    figure_paths['interval_width_histogram'] = _save(fig, 'interval_width_histogram.png')

    groups = [pred.loc[pred['prod_bin']==b, 'abs_error'].dropna().values for b in BIN_ORDER]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.boxplot(groups, labels=BIN_ORDER, showfliers=False); ax.set_title('Absolute error by production bin')
    ax.set_ylabel('|y_true - y_pred_mean| [W]'); plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    figure_paths['absolute_error_by_bin_boxplot'] = _save(fig, 'absolute_error_by_bin_boxplot.png')

    groups = [pred.loc[pred['prod_bin']==b, 'interval_width'].dropna().values for b in BIN_ORDER]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.boxplot(groups, labels=BIN_ORDER, showfliers=False); ax.set_title('Interval width by production bin')
    ax.set_ylabel('interval width [W]'); plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    figure_paths['interval_width_by_bin_boxplot'] = _save(fig, 'interval_width_by_bin_boxplot.png')

In [ ]:
if bins is not None:
    fig, axes = plt.subplots(1, 3, figsize=(13,4))
    for ax, col in zip(axes, ['picp','mpiw','nmpil']):
        ax.bar(bins['bin'], bins[col]); ax.set_title(col + ' by bin')
        plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
    fig.tight_layout()
    figure_paths['picp_mpiw_nmpil_by_bin'] = _save(fig, 'picp_mpiw_nmpil_by_bin.png')

if unc is not None:
    cols = [c for c in ['mae_ratio_vs_normal','std_ratio_vs_normal','mpiw_ratio_vs_normal','picp_delta_vs_normal'] if c in unc.columns]
    fig, ax = plt.subplots(figsize=(11,5))
    x = np.arange(len(unc)); w = 0.8/max(len(cols),1)
    for i, c in enumerate(cols):
        ax.bar(x + i*w, unc[c], width=w, label=c)
    ax.set_xticks(x + w*(len(cols)-1)/2); ax.set_xticklabels(unc['category'], rotation=30, ha='right')
    ax.axhline(1.0, color='k', ls=':', lw=0.8); ax.legend(); ax.set_title('Uncertainty response vs normal')
    figure_paths['uncertainty_response_ratios'] = _save(fig, 'uncertainty_response_ratios.png')
print('figures:', list(figure_paths))

## 8. Manual sweep (sequential, in-notebook)

In [ ]:
SWEEP_CONFIGS = [
    {'name': 'sde_in005e4_out05_ood010', 'sde_proxy_in_weight': 0.00005,
     'sde_proxy_out_weight': 0.5, 'sde_proxy_std_min_ood': 0.10,
     'anomaly_noise_std': 0.05, 'anomaly_noise_prob': 0.7},
    {'name': 'sde_in1e4_out03_ood008', 'sde_proxy_in_weight': 0.0001,
     'sde_proxy_out_weight': 0.3, 'sde_proxy_std_min_ood': 0.08,
     'anomaly_noise_std': 0.03, 'anomaly_noise_prob': 0.7},
]

RUN_SWEEP = False
sweep_summary = []
for name, cfg in pipe.iter_manual_sweep(BASE_CONFIG, SWEEP_CONFIGS):
    od = pipe.make_out_dir(cfg)
    tcmd = pipe.build_train_command(cfg, out_dir=od, run_name=name,
        pvgis_dir=PVGIS_DIR, test_anomaly_scores=TEST_ANOMALY_SCORES,
        train_anomaly_scores=TRAIN_ANOMALY_SCORES, device='cuda', use_wandb=True)
    acmd = pipe.build_analysis_command(od, cfg)
    print('==', name, '->', od)
    if RUN_SWEEP:
        subprocess.run(tcmd, check=True)
        subprocess.run(acmd, check=True)
        sweep_summary.append({'name': name, 'out_dir': od, **pipe.read_posthoc_summary(od)})
if sweep_summary:
    display(pd.DataFrame(sweep_summary))

## 9. Real W&B sweep

In [ ]:
sweep_parameters = {
    'sde_proxy_in_weight': {'values': [0.00005, 0.0001]},
    'sde_proxy_out_weight': {'values': [0.3, 0.5]},
    'sde_proxy_std_min_ood': {'values': [0.08, 0.10]},
    'anomaly_noise_std': {'values': [0.03, 0.05]},
    'anomaly_noise_prob': {'values': [0.7]},
}
sweep_config = pipe.make_sweep_config(sweep_parameters)
sweep_config

In [ ]:
CREATE_SWEEP = False  # set True to register the sweep on W&B
if CREATE_SWEEP:
    import wandb
    sweep_id = wandb.sweep(sweep_config, project=pipe.WANDB_PROJECT, entity=pipe.WANDB_ENTITY)
    print('sweep_id:', sweep_id)
    print('wandb agent ' + pipe.WANDB_ENTITY + '/' + pipe.WANDB_PROJECT + '/' + str(sweep_id))
else:
    print('Set CREATE_SWEEP=True to register. The agent runs the program in sweep_config:')
    print('  ' + pipe.SWEEP_MEMBER_SCRIPT)
    print('  wandb agent ' + pipe.WANDB_ENTITY + '/' + pipe.WANDB_PROJECT + '/<sweep_id>')

## 10. Log post-hoc results to W&B (optional)

Safe no-op if W&B is unavailable. The sweep wrapper does this automatically per run.

In [ ]:
LOG_TO_WANDB = False
if LOG_TO_WANDB:
    import wandb
    run = wandb.init(project=pipe.WANDB_PROJECT, entity=pipe.WANDB_ENTITY, name=run_name, reinit=True)
    summary = pipe.read_posthoc_summary(out_dir)
    print(summary)
    run.log(summary); run.summary.update(summary)
    for label, path in (figure_paths if 'figure_paths' in dir() else {}).items():
        try:
            run.log({'figures/' + label: wandb.Image(str(path))})
        except Exception as e:
            print('skip figure', label, e)
    run.finish()
else:
    print('LOG_TO_WANDB is False. posthoc summary preview:')
    if Path(out_dir, 'sharpness_overview.csv').exists():
        print(pipe.read_posthoc_summary(out_dir))